# Distant Listening Corpus harmonies → BOPP

Goal: load `distant_listening_corpus.datapackage.json` with **dimcat**, derive **BOPP** schemata for the `.expanded.tsv` facet (harmony annotations) from the descriptor schema, represent the harmony labels of one example piece as a BOPP `Annotation`, and export it to every BOPP format: **msgpack**, **JSON**, and **CSV**.

Steps:

1. Load the datapackage with dimcat
2. Read the descriptor schema: payload columns (`label` onwards) + coordinate columns
3. Load the `expanded` facet and pick one example piece
4. Build the BOPP payload (`dcml_harmony`) and extent (`quarter_interval`, unit = quarter notes)
5. Export to msgpack, JSON, and CSV, verifying each round-trip

In [1]:
import json
from pathlib import Path

from dimcat import Dataset

ROOT = Path.cwd()  # this notebook lives in the repo root, next to the descriptor
ds = Dataset()
ds.load(str(ROOT / "distant_listening_corpus.datapackage.json"))
pkg = ds.inputs.get_package("distant_listening_corpus")
print(pkg.resource_names)

/home/laser/git/bopp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['distant_listening_corpus.measures', 'distant_listening_corpus.notes', 'distant_listening_corpus.expanded', 'distant_listening_corpus.chords', 'distant_listening_corpus.metadata']


In [2]:
# Use the schema from the JSON descriptor (not hardcoded knowledge):
# payload schema <- all columns starting from "label" onwards.
descriptor = json.loads((ROOT / "distant_listening_corpus.datapackage.json").read_text())
expanded = next(r for r in descriptor["resources"] if r["name"] == "distant_listening_corpus.expanded")
fields = expanded["schema"]["fields"]
names = [f["name"] for f in fields]
print(f"{len(names)} descriptor fields, primary key: {expanded['schema']['primaryKey']}")
payload_fields = fields[names.index("label"):]
print("payload columns (from 'label' onwards):", [f["name"] for f in payload_fields])
for f in payload_fields:
    print(f"- {f['name']}: {f['type']} — {f.get('title', '')}: {f.get('description', '')}")

36 descriptor fields, primary key: ['corpus', 'piece', 'i']
payload columns (from 'label' onwards): ['label', 'globalkey', 'localkey', 'pedal', 'chord', 'special', 'numeral', 'form', 'figbass', 'changes', 'relativeroot', 'cadence', 'phraseend', 'chord_type', 'globalkey_is_minor', 'localkey_is_minor', 'chord_tones', 'added_tones', 'root', 'bass_note', 'alt_label', 'volta', 'placement']
- label: string — Label: String corresponding to the entire annotation label.
- globalkey: string — Global Key: The key of the entire piece as note name, lowercase designating a minor key.
- localkey: string — Local Key: The key that a Roman numeral is relative to, expressed as a Roman numeral relative to the global key.
- pedal: string — Pedal Point: Specified as Roman numeral.
- chord: string — Chord: The part of an annotation label that corresponds to the actual chord label.
- special: string — Special Label: Column where special labels such as "Fr6" or "Ger65" are stored which are converted internally

In [3]:
harmonies = pkg.get_resource_by_name("distant_listening_corpus.expanded")
df = harmonies.df
print(type(df), df.shape)
print("index levels:", df.index.names)
print("pieces:", len(df.index.droplevel("i").unique()))
df.head(3)

<class 'pandas.core.frame.DataFrame'> (242867, 33)
index levels: ['corpus', 'piece', 'i']
pieces: 1283


mc mn quarterbeats quarterbeats_all_endings  \
corpus piece        i                                                
ABC    n01op18-1_01 0   1  1            0                        0   
                    1   2  2            3                        3   
                    2   3  3            6                        6   

                       duration_qb mc_onset mn_onset timesig  staff  voice  \
corpus piece        i                                                        
ABC    n01op18-1_01 0          3.0        0        0     3/4      4      1   
                    1          3.0        0        0     3/4      4      1   
                    2          3.0        0        0     3/4      4      1   

                       ... chord_type globalkey_is_minor localkey_is_minor  \
corpus piece        i  ...                                                   
ABC    n01op18-1_01 0  ...          M              False             False   
                    1  ...          M              False             False   
                    2  ...          M              False             False   

                      chord_tones added_tones root bass_note alt_label volta  \
corpus piece        i                                                          
ABC    n01op18-1_01 0   (0, 4, 1)          ()    0         0      <NA>  <NA>   
                    1   (1, 5, 2)          ()    1         1      <NA>  <NA>   
                    2   (0, 4, 1)          ()    0         0      <NA>  <NA>   

                      placement  
corpus piece        i            
ABC    n01op18-1_01 0      <NA>  
                    1      <NA>  
                    2      <NA>  

[3 rows x 33 columns]

In [4]:
# One BOPP Annotation addresses one timeline, so demonstrate with a single example piece
# (rather than interleaving all 1283 pieces of the corpus).
corpus, piece = df.index.droplevel("i").unique()[0]
print("example piece:", corpus, "/", piece)
piece_df = df.loc[(corpus, piece)]
print(piece_df.shape)
print("quarterbeats_all_endings element type:", type(piece_df["quarterbeats_all_endings"].iloc[0]))
print("duration_qb dtype:", piece_df["duration_qb"].dtype)

# Extents: quarterbeats_all_endings fractions -> float "start", duration_qb -> "duration"
starts = [float(v) for v in piece_df["quarterbeats_all_endings"].tolist()]
durations = [float(v) for v in piece_df["duration_qb"].tolist()]
print("starts[:5]:", starts[:5])
print("durations[:5]:", durations[:5])

example piece: ABC / n01op18-1_01
(405, 33)
quarterbeats_all_endings element type: <class 'fractions.Fraction'>
duration_qb dtype: float64
starts[:5]: [0.0, 3.0, 6.0, 9.0, 15.0]
durations[:5]: [3.0, 3.0, 3.0, 6.0, 3.0]


## BOPP schemata (generated from the descriptor)

- **Payload** `schemas/v1/payload/dcml_harmony.json` (`payload_type: dcml_harmony`): one parallel array per descriptor column from `label` onwards (23 arrays). Descriptions and item types mirror the descriptor; `chord_tones`/`added_tones` are stored as variable-length integer arrays (the descriptor declares strings, but dimcat materializes tuples of fifth-intervals).
- **Extent** `schemas/v1/extent/quarter_interval.json` (`extent_type: quarter_interval`): `start` + `duration` as float32 arrays.

On the unit question: **yes, quarters can be indicated**. BOPP extent structs have no separate `unit` field — the existing `time_interval` extent hardcodes seconds in its field descriptions — so the idiomatic BOPP way is a dedicated tagged extent type whose tag (`quarter_interval`) and descriptions pin the unit to quarter notes. Fallback, if we ever need plain-JAMS-style interop, is reusing `time_interval` and pretending the quarter coordinates are seconds.

Both schemata are registered in `payloads.json` / `extents.json`, and the msgspec models under `bopp/models/v1/` were regenerated with the repo's codegen (`hatch run codegen:build` equivalent).

In [5]:
import math
from fractions import Fraction

import msgspec
import numpy as np
import pandas as pd

from bopp.io import decode_arrow
from bopp.models.v1.annotation import Annotation

PAYLOAD_COLUMNS = [
    "label", "globalkey", "localkey", "pedal", "chord", "special",
    "numeral", "form", "figbass", "changes", "relativeroot", "cadence",
    "phraseend", "chord_type", "globalkey_is_minor", "localkey_is_minor",
    "chord_tones", "added_tones", "root", "bass_note", "alt_label",
    "volta", "placement",
]
assert PAYLOAD_COLUMNS == [f["name"] for f in payload_fields], "descriptor drift!"


def _scalar(v):
    """Normalize one pandas cell to msgpack/Arrow-friendly plain Python."""
    if v is None or v is pd.NA:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, tuple):  # chord_tones / added_tones
        return [_scalar(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, Fraction):
        return float(v)
    return v


def col_values(s: pd.Series) -> list:
    return [_scalar(v) for v in s.tolist()]


payload = {"payload_type": "dcml_harmony"}
for col in PAYLOAD_COLUMNS:
    payload[col] = col_values(piece_df[col])

annotation = msgspec.convert(
    {
        "media_id": f"dlc:{corpus}/{piece}",
        "bopp_version": "1.0.0",
        "extent": {"extent_type": "quarter_interval", "start": starts, "duration": durations},
        "payload": payload,
    },
    type=Annotation,
    dec_hook=decode_arrow,
)
print(type(annotation.payload).__name__, "|", type(annotation.extent).__name__)
print("n observations:", len(annotation.extent.start))

DcmlHarmonyPayload | QuarterIntervalExtent
n observations: 405


In [6]:
from bopp.util import to_dataframe

bdf = to_dataframe(annotation)
print(bdf.shape)
bdf[["extent:quarter_interval:start", "extent:quarter_interval:duration",
     "payload:dcml_harmony:label", "payload:dcml_harmony:globalkey",
     "payload:dcml_harmony:chord", "payload:dcml_harmony:chord_tones"]].head(8)

(405, 25)


,extent:quarter_interval:start,extent:quarter_interval:duration,payload:dcml_harmony:label,payload:dcml_harmony:globalkey,payload:dcml_harmony:chord,payload:dcml_harmony:chord_tones
0,0.0,3.0,F.I,F,I,"[0, 4, 1]"
1,3.0,3.0,V,F,V,"[1, 5, 2]"
2,6.0,3.0,I,F,I,"[0, 4, 1]"
3,9.0,6.0,IV6,F,IV6,"[3, 0, -1]"
4,15.0,3.0,V65,F,V65,"[5, 2, -1, 1]"
5,18.0,1.0,I,F,I,"[0, 4, 1]"
6,19.0,1.0,vi,F,vi,"[3, 0, 4]"
7,20.0,1.0,ii6,F,ii6,"[-1, 3, 2]"


In [7]:
from bopp.io import encode_arrow, load_bopp_msgpack, save_bopp_msgpack

out = ROOT / "dlc_example_harmony.bopp.msgpack"
save_bopp_msgpack(annotation, str(out))

reloaded = load_bopp_msgpack(str(out))
blob1 = msgspec.msgpack.encode(annotation, enc_hook=encode_arrow)
blob2 = msgspec.msgpack.encode(reloaded, enc_hook=encode_arrow)
assert blob1 == blob2, "msgpack round-trip mismatch"
print(f"round-trip OK ({len(blob1)} bytes); first label: {reloaded.payload.label.to_pylist()[0]!r}, "
      f"starts at quarter {reloaded.extent.start.to_pylist()[0]}")

Successfully saved to /home/laser/git/bopp/dlc_example_harmony.bopp.msgpack (23911 bytes)
round-trip OK (23911 bytes); first label: 'F.I', starts at quarter 0.0


In [8]:
from bopp.io import load_bopp_json, save_bopp_json

save_bopp_json(annotation, str(ROOT / "dlc_example_harmony.bopp.json"))
from_json = load_bopp_json(str(ROOT / "dlc_example_harmony.bopp.json"))
assert msgspec.msgpack.encode(from_json, enc_hook=encode_arrow) == msgspec.msgpack.encode(
    annotation, enc_hook=encode_arrow
)
print("JSON round-trip byte-identical")

Successfully saved to /home/laser/git/bopp/dlc_example_harmony.bopp.json (50306 bytes)
JSON round-trip byte-identical


In [9]:
from bopp.io import load_bopp_csv, to_csv

to_csv(annotation, str(ROOT / "dlc_example_harmony.bopp.csv"))
from_csv = load_bopp_csv(str(ROOT / "dlc_example_harmony.bopp.csv"))

d1, d2 = to_dataframe(annotation), to_dataframe(from_csv)
assert d1.shape == d2.shape and list(d1.columns) == list(d2.columns)


def _cells(s):
    out = []
    for x in s.tolist():
        x = x.as_py() if hasattr(x, "as_py") else (x.tolist() if isinstance(x, np.ndarray) else x)
        out.append(None if (x is None or (isinstance(x, float) and math.isnan(x))) else x)
    return out


def _canon(v):
    # CSV is untyped: nullable ints come back as floats, digit strings as ints.
    if v is None or isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        try:
            return float(v)
        except ValueError:
            return v
    return v


for col in d1.columns:
    assert [_canon(v) for v in _cells(d1[col])] == [_canon(v) for v in _cells(d2[col])], col
print(f"CSV round-trip OK: {d2.shape[0]} rows x {d2.shape[1]} columns, all values equal")

CSV round-trip OK: 405 rows x 25 columns, all values equal


## Notes / open questions

- **Scope**: one `Annotation` per piece (BOPP validates parallel equal-length arrays against a single timeline). Extending this to all 1283 pieces is a loop emitting one file per piece sharing the schemata — say the word if you want that batch export.
- **Nulls**: sparse DCML columns (`pedal`, `special`, `alt_label`, `volta`, `placement`, ...) become Arrow nulls; msgpack/JSON round-trips preserve them exactly.
- **`chord_tones` / `added_tones`**: stored as variable-length integer arrays (fifth-intervals relative to the local tonic). Alternative would be opaque strings — easy to switch if you prefer descriptor-literal fidelity.
- **Formats**: msgpack and JSON round-trips are byte-identical. CSV is untyped, so column *types* do not survive (nullable ints come back as floats, all-digit strings as ints) — *values* are preserved. Two small `bopp/io.py` fixes were needed to make CSV work at all: `to_csv` serializes list-like cells as valid Python literals, and `read_bopp_csv` maps empty fields back to nulls.
- **Environment**: this notebook runs on the local `.venv` (Python 3.12), matching the repo's codegen target (PEP 695 `type` aliases need ≥ 3.12).